In [ ]:
#pratice hitl by langgraph
import os
#llm

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()
llm=ChatOpenAI(model='gpt-4o-mini')

In [ ]:
from langchain_core.tools import tool
import requests
from langchain_tavily import TavilySearch
search_tool = TavilySearch(
    api_key=os.getenv('TAVILY_API_KEY'),
    max_results=3
)

@tool
def get_stock_info(symbol:str) -> dict:
    """
    Recives info about the company or any insitute in the symbol (e.g. 'AAPL', 'TSLA') 
    and fetch the stock  return back the 
    stock details of the company or any insitute
    """
    api_key=os.getenv('ALPHAVANTAGE_API_KEY')
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"
    )
    response=requests.get(url)
    
    return response.json()
    

In [ ]:
from langgraph.types import interrupt,Command
@tool
def purchase_stock(symbol:str,amount:int) ->dict:
    """
    Recives the symbol of company or any insitute (e.g. 'AAPL', 'TSLA') and
    no of shares to purchase 
    -HITL feature here decides whether to buy or reject the response
    """
    
    response=interrupt(f"Are you sure to buy{amount} shares of the {symbol}? y/n")
    if isinstance(response,str)and response.strip().lower() in ['y','yes','sure','whynot']:
        return {
            'status':'sucesss',
            'info':f"The {amount} share of {symbol} has been purchase sucessfully",
            'symbol':symbol,
            'quantity':amount
        }
    else:
        return{
            'status':'negative',
            'info':f"The {amount} share of {symbol}  purchase is canceled by the user",
            'symbol':symbol,
            'quantity':amount
            
        }

In [ ]:
from langgraph.prebuilt import ToolNode,tools_condition

In [ ]:
tools=[get_stock_info,purchase_stock,search_tool]

In [ ]:
llm_with_tools=llm.bind_tools(tools)

In [ ]:
from langgraph.graph import StateGraph,START
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.graph.message import add_messages
class HitlState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [ ]:
def chat_node(state:HitlState):
    
    messages=state['messages']
    response=llm_with_tools.invoke(messages)
    
    return {'messages':response.content}

tool_node=ToolNode(tools)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
checkpointer=MemorySaver()

graph=StateGraph(HitlState)

graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)

graph.add_edge(START,'chat_node')
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')

chatbot=graph.compile(checkpointer=checkpointer)


In [ ]:
chatbot

In [ ]:
print("Write exit or quit to end chat")
while True:
    
    config={'configurable':{'thread_id':'thread_123'}}
    user_input = input("You: ")
    if user_input.lower().strip() in {"exit", "quit"}:
        print("Goodbye!")
        break

    # Build initial state for this turn
    state = {"messages": [HumanMessage(content=user_input)]}

    # Run the graph (may hit an interrupt)
    result = chatbot.invoke(
        state, #type:ignore
        config=config  #type:ignore
    )
    intterupts=result.get("__interrupt__",[])
    if intterupts:
        # Our interrupt payload is the string we passed to interrupt(...)
        prompt_to_human=intterupts[0].value
        print(f'HITL:{prompt_to_human}')
        decison=input("Your decison").strip().lower()
        
        # Resume graph with the human decision ("yes" / "no" / whatever)
        
        result=chatbot.invoke(
            Command(resume=decison),
            config=config #type:ignore
        )
        # Get the latest message from the assistant
        messages = result["messages"]
        last_msg = messages[-1]
        print(f"Bot: {last_msg.content}\n")
        
    

Write exit or quit to end chat
